In [30]:
#Very first step is to import these modules and libraries into the satprocessing kernel
import numpy as np
import dask as ds
import earthaccess as ea
import xarray as xr
import sys
import matplotlib.pyplot as plt
import requests
import pandas as pd
import PyCO2SYS as pyco2
import cartopy
from shapely.ops import unary_union
import cartopy.feature as cfeature
from datetime import date
from scipy.spatial import KDTree
from datasets import Dataset
print("module import success")

import sys
sys.path.append(r'C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\RESOURCES\\python\\utilities\\bootstrap\\')
print("path import success")

import pprint
from environment import bootstrap_environment  
pprint.pprint(bootstrap_environment(preferred="edmund_gable"))
print("success")

from utilities.bootstrap.environment import bootstrap_environment
env = bootstrap_environment(preferred="edmund.gable")
env
print("utilities import success")

#Before this next step, install the library 'resource': conda install -c conda-forge resource
from utilities import get_dates, get_source_file_dates, find_text
find_text("stats")

from utilities import get_prod_files 
get_prod_files("DO", period="DOY")
#"DOY" is climatological 'day of year
#'W' is climatological weekly files

from pathlib import Path

root = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026")
raw_data = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data")
model_data = Path(r"Z:\\MOM6\Hindcasts\\BTM-O2-REGRID")
processed_data=Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Processed_Data\comparison")




module import success
path import success
⚠ Unrecognized preferred label 'edmund_gable'. Valid options: ['NECMAC04363461.local', 'nefscsatdata', 'guihyde', 'Mac.localdomain', 'gdavis', 'egable']
{'dataset_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/DATASETS'),
 'hostname': 'NECL04363598',
 'lookuptable_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/lookuptables'),
 'metadata_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/metadata'),
 'project_root': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'python_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'satlogs_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/logs/satprocessing'),
 'timestamp': datetime.datetime(2026, 6, 16, 12, 53, 58, 967926),
 'utilities_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RE

In [51]:
#load the MOM6-COBALT array
dsg=xr.open_dataset(r"Z:\MOM6\Hindcasts\BTM-O2-REGRID\2005\03\CEFI-BTM_O2-HINDCAST-20050321.nc")
dsg
y_indices=xr.DataArray(np.arange(len(dsg.lat)),dims=["lat"])
x_indices=xr.DataArray(np.arange(len(dsg.lon)),dims=['lon'])

#broadcat x and y coordinates from lon and lat, and put into lat,lon format
y_grid, x_grid = xr.broadcast(y_indices, x_indices)

grid_ids=xr.apply_ufunc(
    lambda y,x: f'{y},{x}',
    y_grid, x_grid,
    vectorize=True,
    dask="allowed"
)

#create the grid id     
grid_ds=dsg.assign_coords(grid_id=(['lat','lon'], grid_ids.values))
print(grid_ds)

<xarray.Dataset> Size: 5MB
Dimensions:  (time: 1, lat: 413, lon: 385)
Coordinates:
  * time     (time) datetime64[ns] 8B 2005-03-21T12:00:00
  * lat      (lat) float64 3kB 22.53 22.59 22.65 22.71 ... 48.25 48.31 48.37
  * lon      (lon) float64 3kB -82.47 -82.39 -82.31 ... -51.65 -51.57 -51.49
    grid_id  (lat, lon) <U7 4MB '0,0' '0,1' '0,2' ... '412,383' '412,384'
Data variables:
    btm_o2   (time, lat, lon) float32 636kB ...
Attributes: (12/34)
    NumFilesInSet:          1
    title:                  NWA12_COBALT_2025_04_bugfixes
    associated_files:       areacello: 19930101.ocean_static.nc
    grid_type:              regular
    grid_tile:              N/A
    external_variables:     areacello
    ...                     ...
    Conventions:            CF-1.0
    History:                Translated to CF-1.0 Conventions by Netcdf-Java C...
    geospatial_lat_min:     22.494149096919973
    geospatial_lat_max:     48.40498164398673
    geospatial_lon_min:     -82.50882070524109
 

In [49]:
#define these functions:

def classify_DO(df, DO_column, battery_col=None):
  clean_df=df[df[DO_column]>0].copy()
  classified_df = clean_df.copy()

  classified_df.loc[classified_df[DO_column] >= 3.9, 'DO_status'] = 'Low Oxygen'
  classified_df.loc[classified_df[DO_column] > 5.0, 'DO_status'] = 'Oxygenated' 
  classified_df.loc[classified_df[DO_column] < 3.9, 'DO_status'] = 'Mildly Hypoxic'
  classified_df.loc[classified_df[DO_column] < 2.41, 'DO_status'] = 'Hypoxia'
  classified_df.loc[classified_df[DO_column] < 0.2, 'DO_status'] = 'Anoxia'
    
  return classified_df
  
def open_file(file):
  ds=xr.open_dataset(file)
  df=ds.to_dataframe().reset_index()
  print(df.columns.to_list())
  return df

def check_units(ds, DO_col):
  print(ds[DO_col].attrs)  

#Use this function to match observational coordinates to MOM6 coordinates 
def prepare_df(df, DO_col, id_col, obs_depth_col, bottom_depth_col, lat_col, lon_col, time_col, mom6_grid_df, battery_col=None):
  
  # drop values that are nan
  df=df.dropna(subset=[DO_col])


  # make sure measurement is at bottom
  depth_mismatch = abs(df[obs_depth_col] - df[bottom_depth_col])
  depth_mismatch=depth_mismatch.astype(float)

  max_mismatch=np.minimum(0.1*df[bottom_depth_col], 20)
  df_filtered=df[depth_mismatch<max_mismatch].copy()

  # match to MOM6 grid timestamp
  df_filtered["matched_time"] = pd.to_datetime(df_filtered[time_col]).dt.normalize() + pd.Timedelta(hours=12)
 
  # match to spatial grids
  pt_lats = xr.DataArray(df_filtered[lat_col], dims='points')
  pt_lons = xr.DataArray(df_filtered[lon_col], dims='points')

  # Query the existing dataset directly for the nearest coordinates
  nearest_grid = grid_ds.sel(lat=pt_lats, lon=pt_lons, method='nearest')

  # Assign those exact grid coordinates back to the dataframe
  #df_filtered['grid_lat'] = nearest_grid.lat.values
  #df_filtered['grid_lon'] = nearest_grid.lon.values
  df_filtered['matched_lat'] = nearest_grid.lat.values
  df_filtered['matched_lon'] = nearest_grid.lon.values
  df_filtered['grid_id'] = nearest_grid.grid_id.values
  df=df_filtered
  # Force grid_id to remain text format so commas aren't swallowed
  df['grid_id'] = df['grid_id'].astype(str)
  rename_dict = {
        DO_col: "dissolved_oxygen",
        id_col: "id",
        obs_depth_col: "obs_depth",
        bottom_depth_col: "bottom_depth",
        lat_col: "lat",
        lon_col: "lon",
        time_col: "time",
    }
  df = df.rename(columns=rename_dict)
  #take everything in one grid cell of point data and average it
  agg_rules = {
        "dissolved_oxygen": "mean",    # Average the oxygen measurements in the cell
        "obs_depth": "mean",           # Average the depths they were taken at
        "bottom_depth": "mean",        # Average the seafloor depths
        "lat": "mean",                 # Average the exact vessel latitudes
        "lon": "mean",                 # Average the exact vessel longitudes
        "id": "first",                 # Keep the cruise/vessel ID 
        #"time": "first",               # DON'T Keep the original raw timestamp
        "matched_lat": "first",        # Grid lats are identical for this group, keep first
        "matched_lon": "first"         # Grid lons are identical for this group, keep first
    }
    

#Group by the day and grid cell, apply rules, and restore flat dataframe structure
  df = df.groupby(['matched_time', 'grid_id'], as_index=False).agg(agg_rules)
    
  df = classify_DO(df, 'dissolved_oxygen', battery_col=battery_col)
  requested_columns = [
        "id",
        "lat",
        "lon",
        "obs_depth",
        "bottom_depth",
        "dissolved_oxygen",
        "DO_status",  
        "grid_id",
        "matched_time",
        "matched_lat",
        "matched_lon"
        ]

 
  return df[requested_columns]

#Use this instead for data that does not have depth (eg. eMOLT)
def prepare_df_no_depth(df, DO_col, id_col, lat_col, lon_col, time_col, mom6_grid_df, battery_col=None):
    # 1. Drop values that are nan (explicitly copy to avoid SettingWithCopy warnings)
    df_filtered = df.dropna(subset=[DO_col]).copy()

    # 2. FILTER BATTERY (Only if a battery column name was provided)
    if battery_col is not None:
        df_filtered = df_filtered[df_filtered[battery_col] >= 50]
        
    df_filtered = df_filtered.reset_index(drop=True)
    
    # 3. Match to MOM6 grid timestamp
    df_filtered["matched_time"] = pd.to_datetime(df_filtered[time_col]).dt.normalize() + pd.Timedelta(hours=12)
     
    # 4. Match to spatial grids
    pt_lats = xr.DataArray(df_filtered[lat_col], dims='points')
    pt_lons = xr.DataArray(df_filtered[lon_col], dims='points')

    # Query the existing dataset directly for the nearest coordinates
    # (Note: ensure 'dsg' is defined globally or passed into the function)
    nearest_grid = grid_ds.sel(lat=pt_lats, lon=pt_lons, method='nearest')

    # Assign those exact grid coordinates back to the dataframe
    df_filtered['matched_lat'] = nearest_grid.lat.values
    df_filtered['matched_lon'] = nearest_grid.lon.values
    df_filtered['grid_id'] = nearest_grid.grid_id.values.astype(str)
     # Force grid_id to remain text format so commas aren't swallowed
    df['grid_id'] = df['grid_id'].astype(str)
    # 5. Rename Columns
    rename_dict = {
        DO_col: "dissolved_oxygen",
        id_col: "id",
        lat_col: "lat",
        lon_col: "lon",
        time_col: "time",
    }
    df_mapped = df_filtered.rename(columns=rename_dict)
    
    # 6. Aggregate data by grid cell and day
    agg_rules = {
        "dissolved_oxygen": "mean",     # Average the oxygen measurements in the cell
        "lat": "mean",                  # Average the exact vessel latitudes
        "lon": "mean",                  # Average the exact vessel longitudes
        "id": "first",                  # Keep the cruise/vessel ID 
        "matched_lat": "first",         # Grid lats are identical for this group
        "matched_lon": "first"          # Grid lons are identical for this group
    }
    
    df_agg = df_mapped.groupby(['matched_time', 'grid_id']).agg(agg_rules).reset_index()
    
    # 7. Classify DO (Pass 'dissolved_oxygen' because columns were renamed)
    df_classified = classify_DO(df_agg, 'dissolved_oxygen', battery_col=battery_col)
    
    # 8. Filter to requested columns
    requested_columns = [
        "id",
        "lat",
        "lon",
        "dissolved_oxygen",
        "DO_status",  
        "grid_id",
        "matched_time",
        "matched_lat",
        "matched_lon"
    ]
    
    return df_classified[requested_columns]

NetCDF's like MOM6 have grid index tuples like (x, y) to indicate their position in a gridded latitude-longitude map. We can access them using xarray. I will be doing this to access a three by three grid of the MOM6-COBALT data around an observational point in order to identify the mean around that point, the stdev, the minumum, the maximumm and the grid with the value of bottom O2 closest to the observational O2 

In [6]:
ds=xr.open_dataset(r"Z:\MOM6\Hindcasts\BTM-O2-REGRID\2002\01\CEFI-BTM_O2-HINDCAST-20020101.nc")
print(ds[['lat','lon']])

<xarray.Dataset> Size: 6kB
Dimensions:  (lat: 413, lon: 385)
Coordinates:
  * lat      (lat) float64 3kB 22.53 22.59 22.65 22.71 ... 48.25 48.31 48.37
  * lon      (lon) float64 3kB -82.47 -82.39 -82.31 ... -51.65 -51.57 -51.49
Data variables:
    *empty*
Attributes: (12/34)
    NumFilesInSet:          1
    title:                  NWA12_COBALT_2025_04_bugfixes
    associated_files:       areacello: 19930101.ocean_static.nc
    grid_type:              regular
    grid_tile:              N/A
    external_variables:     areacello
    ...                     ...
    Conventions:            CF-1.0
    History:                Translated to CF-1.0 Conventions by Netcdf-Java C...
    geospatial_lat_min:     22.494149096919973
    geospatial_lat_max:     48.40498164398673
    geospatial_lon_min:     -82.50882070524109
    geospatial_lon_max:     -51.448697088915196


In [33]:

def load_3x3x3_MOM6(df, model_data, mom6_var='btm_o2'):
    model_data = Path(model_data)
    df_list = []

    df = df.copy()
    df['matched_time'] = pd.to_datetime(df['matched_time'])
    unique_dates = df['matched_time'].dt.normalize().unique()

    print(f"Found {len(unique_dates)} unique dates to query.")

    for center_date in unique_dates:
        # Define the 3-day window range and build a list
        t_minus = center_date - pd.Timedelta(days=1)
        t_plus  = center_date + pd.Timedelta(days=1)
        target_days = [t_minus, center_date, t_plus]
        
        daily_slices = []

        # Open each file individually 
        for d in target_days:
            year = str(d.year)
            month = f"{d.month:02d}"
            day = f"{d.day:02d}"
            
            file_name = f"CEFI-BTM_O2-HINDCAST-{year}{month}{day}.nc"
            target_file_path = model_data / year / month / file_name

            if target_file_path.exists():
                ds = xr.open_dataset(target_file_path)
                # Convert mol/kg units to mg/L 
                o2_converted = ds[mom6_var] * 31998.8 * 1.025
                daily_slices.append(o2_converted)
                ds.close()
                    
        if not daily_slices:
            continue

        print(f"Extracting 3x3x3 space-time matrix for {center_date.strftime('%Y-%m-%d')}...")
        df_day = df[df['matched_time'].dt.normalize() == center_date]

# Combine 3 days into a temporary 3D time-series dataset
        ds_window = xr.concat(daily_slices, dim='time')
        
        # --- FIX: Overwrite ds_window's grid_id with the master grid_ds layout ---
        # This forces the indices to use the exact same positive tracking system
        ds_window = ds_window.assign_coords(grid_id=(['lat', 'lon'], grid_ds.grid_id.values))
        
        # Hard lock coordinate datatypes to string to prevent trailing zero drops
        ds_window['grid_id'] = ds_window['grid_id'].astype(str)
        # Combine 3 days into a temporary 3D time-series dataset
        #ds_window = xr.concat(daily_slices, dim='time')
        
        # Hard lock coordinate datatypes to string to prevent trailing zero drops
        #if 'grid_id' in ds_window.coords:
          #  ds_window['grid_id'] = ds_window['grid_id'].astype(str)

        # reference the flat, 1D coordinates
        lat_arr = ds_window.lat.values
        lon_arr = ds_window.lon.values

        for idx, row in df_day.iterrows():
            # 1. Find the closest target values in the coordinate lists 
            lat_idx = int(np.argmin(np.abs(lat_arr - row['lat'])))
            lon_idx = int(np.argmin(np.abs(lon_arr - row['lon'])))
            
            # 2. Slice out the immediate 3-point neighbors from the 1D arrays 
            lat_neighbors = lat_arr[max(0, lat_idx-1) : min(len(lat_arr), lat_idx+2)]
            lon_neighbors = lon_arr[max(0, lon_idx-1) : min(len(lon_arr), lon_idx+2)]
            
            # 3. Use xarray's native selection to grab the 3x3x3 space-time cube dataset
            cube_3x3x3 = ds_window.sel(lat=lat_neighbors, lon=lon_neighbors)
            
            # Extract the raw numpy array block for analysis
            o2_cube = cube_3x3x3.values # Expected shape: (time, lat, lon)
            
            if np.all(np.isnan(o2_cube)):
                continue

            # Calculate summary window stats 
            mean_o2 = float(np.nanmean(o2_cube))
            std_o2  = float(np.nanstd(o2_cube))
            min_o2  = float(np.nanmin(o2_cube))
            max_o2  = float(np.nanmax(o2_cube))

            # Locate the cell closest to field observations
            abs_diff = np.abs(o2_cube - row['dissolved_oxygen'])
            flat_idx = np.nanargmin(abs_diff) #finds index of smallest vaue in array (ignoring nan)
            grid_t, grid_y, grid_x = np.unravel_index(flat_idx, o2_cube.shape)

            closest_val  = float(o2_cube[grid_t, grid_y, grid_x])
            closest_lat  = float(cube_3x3x3.lat.values[grid_y])
            closest_lon  = float(cube_3x3x3.lon.values[grid_x])
            closest_time = pd.Timestamp(cube_3x3x3.time.values[grid_t])

            # Stop trailing zeros from dropping in grid ID
            if 'grid_id' in cube_3x3x3.coords:
                raw_grid_id = str(cube_3x3x3.grid_id.values[grid_y, grid_x])
                closest_grid_id = raw_grid_id if raw_grid_id.startswith('(') else f"({raw_grid_id})"
            else:
                closest_grid_id = f"({closest_lat},{closest_lon})"

            # Append the row dictionary metrics
            matched_row = row.to_dict()
            matched_row.update({
                "cube_mean": mean_o2,
                "cube_std": std_o2,
                "cube_min": min_o2,
                "cube_max": max_o2,
                "closest_model_val": closest_val,
                "closest_grid_id": closest_grid_id,
                "closest_lat": closest_lat,
                "closest_lon": closest_lon,
                "closest_time": closest_time
            })
            df_list.append(matched_row)

    return pd.DataFrame(df_list).reset_index(drop=True)

In [50]:
def load_3x3x3_MOM6(df, model_data, mom6_var='btm_o2'):
  
    model_data = Path(model_data)
    df_list = []

    df = df.copy()
    df['matched_time'] = pd.to_datetime(df['matched_time'])
    unique_dates = df['matched_time'].dt.normalize().unique()

    print(f"Found {len(unique_dates)} unique dates to query.")

    for center_date in unique_dates:
        # Define the 3-day window range and build a list
        t_minus = center_date - pd.Timedelta(days=1)
        t_plus  = center_date + pd.Timedelta(days=1)
        target_days = [t_minus, center_date, t_plus]
        
        daily_slices = []

        # Open each file individually 
        for d in target_days:
            year = str(d.year)
            month = f"{d.month:02d}"
            day = f"{d.day:02d}"
            
            file_name = f"CEFI-BTM_O2-HINDCAST-{year}{month}{day}.nc"
            target_file_path = model_data / year / month / file_name

            if target_file_path.exists():
                ds = xr.open_dataset(target_file_path)
                
                # --- NEW: Dynamically build the 'grid_id' for each daily file ---
                y_indices = xr.DataArray(np.arange(len(ds.lat)), dims=["lat"])
                x_indices = xr.DataArray(np.arange(len(ds.lon)), dims=["lon"])
                y_grid, x_grid = xr.broadcast(y_indices, x_indices)
                
                grid_ids = xr.apply_ufunc(
                    lambda y, x: f"{y},{x}",
                    y_grid, x_grid,
                    vectorize=True
                )
                ds = ds.assign_coords(grid_id=(['lat', 'lon'], grid_ids.values))
                # ---------------------------------------------------------------

                # Convert mol/kg units to mg/L 
                o2_converted = ds[mom6_var] * 31998.8 * 1.025
                daily_slices.append(o2_converted)
                ds.close()
                    
        if not daily_slices:
            continue

        print(f"Extracting 3x3x3 space-time matrix for {center_date.strftime('%Y-%m-%d')}...")
        df_day = df[df['matched_time'].dt.normalize() == center_date]


# Combine 3 days into a temporary 3D time-series dataset
        ds_window = xr.concat(daily_slices, dim='time')
        
        # --- FIX: Overwrite ds_window's grid_id with the master grid_ds layout ---
        # This forces the indices to use the exact same positive tracking system
        ds_window = ds_window.assign_coords(grid_id=(['lat', 'lon'], grid_ds.grid_id.values))
        
        # Hard lock coordinate datatypes to string to prevent trailing zero drops
        ds_window['grid_id'] = ds_window['grid_id'].astype(str)

        # Combine 3 days into a temporary 3D time-series dataset
        #ds_window = xr.concat(daily_slices, dim='time')
        
        # Hard lock coordinate datatypes to string to prevent trailing zero drops
        #if 'grid_id' in ds_window.coords:
           # ds_window['grid_id'] = ds_window['grid_id'].astype(str)

        # Reference the flat, 1D spatial coordinates for index locating
        lat_arr = ds_window.lat.values
        lon_arr = ds_window.lon.values

        for idx, row in df_day.iterrows():
            # 1. Find the closest target values in the coordinate lists 
            lat_idx = int(np.argmin(np.abs(lat_arr - row['lat'])))
            lon_idx = int(np.argmin(np.abs(lon_arr - row['lon'])))
            
            # 2. Define index slices (Safely switches to positional indexing)
            lat_slice = slice(max(0, lat_idx - 1), lat_idx + 2)
            lon_slice = slice(max(0, lon_idx - 1), lon_idx + 2)
            
            # 3. Use positional indexing (.isel) to safely grab the space-time cube
            cube_3x3x3 = ds_window.isel(lat=lat_slice, lon=lon_slice)
            
            # Extract the raw numpy array block for analysis
            o2_cube = cube_3x3x3.values # Expected shape: (time, lat, lon)
            
            if np.all(np.isnan(o2_cube)):
                continue

            # Calculate summary window stats 
            mean_o2 = float(np.nanmean(o2_cube))
            std_o2  = float(np.nanstd(o2_cube))
            min_o2  = float(np.nanmin(o2_cube))
            max_o2  = float(np.nanmax(o2_cube))

            # Locate the cell closest to field observations
            abs_diff = np.abs(o2_cube - row['dissolved_oxygen'])
            flat_idx = np.nanargmin(abs_diff) 
            grid_t, grid_y, grid_x = np.unravel_index(flat_idx, o2_cube.shape)

            closest_val  = float(o2_cube[grid_t, grid_y, grid_x])
            closest_lat  = float(cube_3x3x3.lat.values[grid_y])
            closest_lon  = float(cube_3x3x3.lon.values[grid_x])
            closest_time = pd.Timestamp(cube_3x3x3.time.values[grid_t])

            # Extract the explicit "y,x" string from our new coordinates
            #if 'grid_id' in cube_3x3x3.coords:
            #    raw_grid_id = str(cube_3x3x3.grid_id.values[f'({grid_y}, {grid_x})'])
            #    closest_grid_id = raw_grid_id if raw_grid_id.startswith('(') else f"({raw_grid_id})"
           # else:
              #  closest_grid_id = f"({closest_lat},{closest_lon})"
            # Extract the explicit "y,x" string from our new coordinates
            if 'grid_id' in cube_3x3x3.coords:
                # 1. Grab the raw "y,x" string value using integer indices
                raw_grid_id = str(cube_3x3x3.grid_id.values[grid_y, grid_x])
    
                # 2. Add the parentheses safely around the string
                closest_grid_id = f"({raw_grid_id})"
            else:
                closest_grid_id = f"({closest_lat},{closest_lon})"

            # Append the row dictionary metrics
            matched_row = row.to_dict()
            matched_row.update({
                "cube_mean": mean_o2,
                "cube_std": std_o2,
                "cube_min": min_o2,
                "cube_max": max_o2,
                "closest_model_val": closest_val,
                "closest_grid_id": closest_grid_id,
                "closest_lat": closest_lat,
                "closest_lon": closest_lon,
                "closest_time": closest_time
            })
            df_list.append(matched_row)

    return pd.DataFrame(df_list).reset_index(drop=True)

In [16]:
model_data = r"Z:\MOM6\Hindcasts\BTM-O2-REGRID"

# Load your observational csv
dfp = pd.read_csv(processed_data / "ECOMON_MOM6_matched_coords.csv")

# Run the query function
final = load_3x3x3_MOM6(dfp, model_data, mom6_var='btm_o2')




Found 1232 unique dates to query.
Extracting 3x3x3 space-time matrix for 2009-08-18...
Extracting 3x3x3 space-time matrix for 2009-08-19...
Extracting 3x3x3 space-time matrix for 2009-08-20...
Extracting 3x3x3 space-time matrix for 2009-08-21...
Extracting 3x3x3 space-time matrix for 2009-08-22...
Extracting 3x3x3 space-time matrix for 2009-08-23...
Extracting 3x3x3 space-time matrix for 2009-08-24...
Extracting 3x3x3 space-time matrix for 2009-08-25...


KeyboardInterrupt: 

In [ ]:
#export separately incase export fails
final.to_csv(processed_data / "ECOMON_MOM6_3x3x3.csv")

Implementation script:

In [52]:
#First, Process your data to match it to MOM6 grids
#Build Master Grid
#dsg = xr.open_dataset(r"Z:\MOM6\Hindcasts\BTM-O2-REGRID\2005\03\CEFI-BTM_O2-HINDCAST-20050321.nc")
#ny, nx = len(dsg.lat), len(dsg.lon)
#grid_id_matrix = np.array([[f"{y},{x}" for x in range(nx)] for y in range(ny)], dtype=str)
#grid_ds = dsg.assign_coords(grid_id=(['lat', 'lon'], grid_id_matrix))

#1. Open the observational data file 
file=(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data\eMOLT\eMOLT_2000_2026.nc")
df=open_file(file)

#check to make sure units are in mg/L 
check_units(df, 'DO')

#2. read in the grid extent
mom6_grid_df=pd.read_csv(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Index\MOM6_Grid_Extent.csv")



#3. prepare and export the prepared dataset, replacing inputs as needed from printed key below
dfp=prepare_df_no_depth(df, 'DO', 'tow_id', 'latitude', 'longitude', 'time', mom6_grid_df, 'water_detect_perc')
#dfp
#dfp.to_csv(processed_data / "eMOLT_matched_coords_2023_2026.csv")

#dfp['grid_id'] = dfp['grid_id'].astype(str)
#dfp.to_csv(processed_data / "eMOLT_matched_coords_2023_2026.csv", index=False)

['row', 'tow_id', 'time', 'latitude', 'longitude', 'temperature', 'DO', 'DO_percentage', 'water_detect_perc', 'sensor_type']
{}


KeyError: 'grid_id'

In [23]:
#Second, run data through the load_3x3x3_MOM6(df, model_data, mom6_var='btm_o2') function
df=pd.read_excel(processed_data / "eMOLT_matched_coords_2023.xlsx")
final=load_3x3x3_MOM6(df, model_data, mom6_var='btm_o2')

Found 107 unique dates to query.
Extracting 3x3x3 space-time matrix for 2023-07-31...
Extracting 3x3x3 space-time matrix for 2023-08-01...
Extracting 3x3x3 space-time matrix for 2023-08-02...
Extracting 3x3x3 space-time matrix for 2023-08-03...
Extracting 3x3x3 space-time matrix for 2023-08-04...
Extracting 3x3x3 space-time matrix for 2023-08-05...
Extracting 3x3x3 space-time matrix for 2023-08-06...
Extracting 3x3x3 space-time matrix for 2023-08-07...
Extracting 3x3x3 space-time matrix for 2023-08-08...
Extracting 3x3x3 space-time matrix for 2023-08-09...
Extracting 3x3x3 space-time matrix for 2023-08-10...
Extracting 3x3x3 space-time matrix for 2023-08-11...
Extracting 3x3x3 space-time matrix for 2023-08-12...
Extracting 3x3x3 space-time matrix for 2023-08-13...
Extracting 3x3x3 space-time matrix for 2023-08-14...
Extracting 3x3x3 space-time matrix for 2023-08-15...
Extracting 3x3x3 space-time matrix for 2023-08-16...
Extracting 3x3x3 space-time matrix for 2023-08-17...
Extracting 3x

In [24]:
#Third, export as its own file
final.to_csv(processed_data / 'eMOLT_MOM6_3x3x3.csv')